In [1]:
import json
import tarfile
import textwrap
from pathlib import Path
from types import SimpleNamespace

import boto3
from botocore.exceptions import ClientError

session = boto3.session.Session()
region = session.region_name or 'us-east-1'
sm = session.client('sagemaker', region_name=region)
s3 = session.client('s3', region_name=region)

print('region:', region)

region: us-east-1


In [2]:
BUCKET = 'agri-price-dev-raw'
ROLE_ARN = 'arn:aws:iam::654654220088:role/LabRole'
FEATURES_PREFIX = 'processed/features/'
PIPELINE_NAME = 'agri-price-train-evaluate-register'
MODEL_PACKAGE_GROUP_NAME = 'agri-price-multi-output'
FRAMEWORK_VERSION = '1.2-1'
CODE_PREFIX = 'sagemaker/code/'

CODE_BUNDLE_S3_URI = f's3://{BUCKET}/{CODE_PREFIX}sagemaker_pipeline_source.tar.gz'
EVALUATE_SCRIPT_S3_URI = f's3://{BUCKET}/{CODE_PREFIX}evaluate.py'
FEATURES_S3_URI = f's3://{BUCKET}/{FEATURES_PREFIX}'

WORKDIR = Path.cwd() / 'agri_price_sagemaker_notebook_workdir'
WORKDIR.mkdir(exist_ok=True)
WORKDIR

PosixPath('/home/sagemaker-user/agri-price/agri_price_sagemaker_notebook_workdir')

In [3]:
phase_c_train_multi_code = textwrap.dedent('''
from __future__ import annotations

import json
import math
import pickle
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline

DEFAULT_MODEL_NAME = "multi_output_xgboost"
TARGET_PREFIX = "target_next_day_"
MANAGED_METADATA_FILES = ("metrics.json", "training_summary.json", "comparison_report.json", "model.pkl")

class PhaseCValidationError(ValueError):
    pass

@dataclass(frozen=True)
class TrainingConfig:
    validation_fraction: float = 0.2
    min_validation_rows: int = 14
    min_training_rows: int = 30
    random_state: int = 42
    seasonal_period: int = 7
    walk_forward_windows: int = 3
    xgb_n_estimators: int = 300
    xgb_max_depth: int = 6
    xgb_learning_rate: float = 0.05
    xgb_subsample: float = 0.9
    xgb_colsample_bytree: float = 0.9
    xgb_reg_lambda: float = 1.0
    target_columns: tuple[str, ...] = ()

def load_features_dataset(features_path):
    path = Path(features_path)
    features = pd.read_parquet(path)
    features = features.copy()
    features['date'] = pd.to_datetime(features['date'], errors='coerce')
    features = features.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    if features.empty:
        raise PhaseCValidationError('Features dataset is empty after loading.')
    return features

def resolve_target_columns(features, config):
    requested = tuple(c for c in config.target_columns if c)
    if requested:
        return requested
    found = tuple(c for c in features.columns if c.startswith(TARGET_PREFIX))
    if not found:
        raise PhaseCValidationError('No target columns found in the features dataset.')
    return found

def prepare_modeling_frame(features, target_columns):
    frame = features.copy().dropna(subset=list(target_columns)).reset_index(drop=True)
    feature_columns = [c for c in frame.columns if c not in ('date', *target_columns)]
    for column in feature_columns:
        if pd.api.types.is_categorical_dtype(frame[column]) or pd.api.types.is_object_dtype(frame[column]):
            frame[column] = pd.to_numeric(frame[column], errors='coerce')
    return frame

def split_time_aware_dataset(features, config):
    validation_rows = max(config.min_validation_rows, math.ceil(len(features) * config.validation_fraction))
    validation_rows = min(validation_rows, len(features) - config.min_training_rows)
    split_index = len(features) - validation_rows
    train_frame = features.iloc[:split_index].copy()
    validation_frame = features.iloc[split_index:].copy()
    return train_frame.reset_index(drop=True), validation_frame.reset_index(drop=True)

def build_walk_forward_splits(features, config):
    splits = []
    validation_rows = max(config.min_validation_rows, math.ceil(len(features) * config.validation_fraction))
    for window_index in range(config.walk_forward_windows, 0, -1):
        end = len(features) - (window_index - 1) * validation_rows
        start = end - validation_rows
        if start <= 0:
            continue
        train_frame = features.iloc[:start].copy()
        validation_frame = features.iloc[start:end].copy()
        if len(train_frame) < config.min_training_rows or len(validation_frame) < config.min_validation_rows:
            continue
        splits.append((train_frame.reset_index(drop=True), validation_frame.reset_index(drop=True)))
    return splits

def create_primary_regressor(config):
    from xgboost import XGBRegressor
    return MultiOutputRegressor(XGBRegressor(n_estimators=config.xgb_n_estimators, max_depth=config.xgb_max_depth, learning_rate=config.xgb_learning_rate, subsample=config.xgb_subsample, colsample_bytree=config.xgb_colsample_bytree, reg_lambda=config.xgb_reg_lambda, random_state=config.random_state, objective='reg:squarederror', n_jobs=1), n_jobs=1)

def fit_primary_model(train_frame, feature_columns, target_columns, config):
    model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('regressor', create_primary_regressor(config))])
    model.fit(train_frame[feature_columns], train_frame[list(target_columns)])
    return model

def predict_multi_output(model, frame, feature_columns, target_columns):
    return pd.DataFrame(model.predict(frame[feature_columns]), columns=list(target_columns), index=frame.index)

def build_persistence_predictions(train_frame, validation_frame, target_columns):
    out = {}
    for column in target_columns:
        history = pd.concat([train_frame[column].reset_index(drop=True), validation_frame[column].reset_index(drop=True)], ignore_index=True)
        predicted = history.shift(1).iloc[len(train_frame):].reset_index(drop=True).fillna(train_frame[column].iloc[-1])
        out[column] = pd.Series(predicted.to_numpy(), index=validation_frame.index, dtype='float64')
    return pd.DataFrame(out, index=validation_frame.index)

def build_seasonal_naive_predictions(train_frame, validation_frame, target_columns, seasonal_period):
    fallback = build_persistence_predictions(train_frame, validation_frame, target_columns)
    out = {}
    for column in target_columns:
        history = pd.concat([train_frame[column].reset_index(drop=True), validation_frame[column].reset_index(drop=True)], ignore_index=True)
        predicted = history.shift(seasonal_period).iloc[len(train_frame):].reset_index(drop=True).fillna(fallback[column].reset_index(drop=True))
        out[column] = pd.Series(predicted.to_numpy(), index=validation_frame.index, dtype='float64')
    return pd.DataFrame(out, index=validation_frame.index)

def build_multi_target_metric_summary(actual_frame, predicted_frame, train_rows, validation_rows):
    by_target = {}
    maes = []
    rmses = []
    for column in actual_frame.columns:
        mae = float(mean_absolute_error(actual_frame[column], predicted_frame[column]))
        rmse = float(math.sqrt(mean_squared_error(actual_frame[column], predicted_frame[column])))
        maes.append(mae)
        rmses.append(rmse)
        by_target[column] = {'mae': mae, 'rmse': rmse}
    return {'overall': {'mae': float(sum(maes) / len(maes)), 'rmse': float(sum(rmses) / len(rmses))}, 'by_target': by_target, 'train_rows': int(train_rows), 'validation_rows': int(validation_rows)}

def summarize_multi_target_fold_metrics(folds, target_columns):
    by_target = {}
    for column in target_columns:
        maes = [fold['by_target'][column]['mae'] for fold in folds]
        rmses = [fold['by_target'][column]['rmse'] for fold in folds]
        by_target[column] = {'mean_mae': float(sum(maes) / len(maes)), 'mean_rmse': float(sum(rmses) / len(rmses))}
    overall_mae = [fold['overall']['mae'] for fold in folds]
    overall_rmse = [fold['overall']['rmse'] for fold in folds]
    return {'folds': folds, 'overall': {'mean_mae': float(sum(overall_mae) / len(overall_mae)), 'mean_rmse': float(sum(overall_rmse) / len(overall_rmse))}, 'by_target': by_target}

def evaluate_models_with_walk_forward(target_columns, feature_columns, splits, config):
    xgb_folds = []
    persistence_folds = []
    seasonal_folds = []
    for fold_number, (train_frame, validation_frame) in enumerate(splits, start=1):
        xgb = predict_multi_output(fit_primary_model(train_frame, feature_columns, target_columns, config), validation_frame, feature_columns, target_columns)
        persistence = build_persistence_predictions(train_frame, validation_frame, target_columns)
        seasonal = build_seasonal_naive_predictions(train_frame, validation_frame, target_columns, config.seasonal_period)
        xgb_folds.append({'fold': fold_number, **build_multi_target_metric_summary(validation_frame[list(target_columns)], xgb, len(train_frame), len(validation_frame))})
        persistence_folds.append({'fold': fold_number, **build_multi_target_metric_summary(validation_frame[list(target_columns)], persistence, len(train_frame), len(validation_frame))})
        seasonal_folds.append({'fold': fold_number, **build_multi_target_metric_summary(validation_frame[list(target_columns)], seasonal, len(train_frame), len(validation_frame))})
    models = {'xgboost': summarize_multi_target_fold_metrics(xgb_folds, target_columns), 'persistence': summarize_multi_target_fold_metrics(persistence_folds, target_columns), 'seasonal_naive': summarize_multi_target_fold_metrics(seasonal_folds, target_columns)}
    ranking = sorted(({'model_name': n, 'mean_mae': d['overall']['mean_mae'], 'mean_rmse': d['overall']['mean_rmse']} for n, d in models.items()), key=lambda item: (item['mean_mae'], item['mean_rmse']))
    return {'evaluation_type': 'walk_forward', 'target_columns': list(target_columns), 'feature_count': len(feature_columns), 'window_count': len(splits), 'models': models, 'ranking_by_mean_mae': ranking, 'winner': ranking[0]['model_name']}

def build_date_range_summary(series):
    return {'start_date': pd.Timestamp(series.min()).strftime('%Y-%m-%d'), 'end_date': pd.Timestamp(series.max()).strftime('%Y-%m-%d')}

def train_phase_c_models(features, config):
    target_columns = resolve_target_columns(features, config)
    frame = prepare_modeling_frame(features, target_columns)
    feature_columns = [c for c in frame.columns if c not in ('date', *target_columns)]
    train_frame, validation_frame = split_time_aware_dataset(frame, config)
    splits = build_walk_forward_splits(frame, config)
    comparison_report = evaluate_models_with_walk_forward(target_columns, feature_columns, splits, config)
    holdout_model = fit_primary_model(train_frame, feature_columns, target_columns, config)
    holdout_predictions = predict_multi_output(holdout_model, validation_frame, feature_columns, target_columns)
    holdout_metrics = build_multi_target_metric_summary(validation_frame[list(target_columns)], holdout_predictions, len(train_frame), len(validation_frame))
    final_model = fit_primary_model(frame, feature_columns, target_columns, config)
    trained_at = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    summary = {'model_name': DEFAULT_MODEL_NAME, 'algorithm': 'sklearn.multioutput.MultiOutputRegressor(xgboost.XGBRegressor)', 'trained_at_utc': trained_at, 'target_columns': list(target_columns), 'feature_columns': feature_columns, 'training_date_range': build_date_range_summary(frame['date']), 'holdout_train_date_range': build_date_range_summary(train_frame['date']), 'holdout_validation_date_range': build_date_range_summary(validation_frame['date']), 'config': asdict(config), 'metrics': holdout_metrics, 'walk_forward_windows': len(splits), 'comparison_report_file': 'comparison_report.json'}
    artifact = {'model': final_model, 'target_columns': list(target_columns), 'feature_columns': feature_columns, 'model_name': DEFAULT_MODEL_NAME, 'trained_at_utc': trained_at, 'config': asdict(config), 'comparison_report': comparison_report}
    return {'artifact_payload': artifact, 'metrics': holdout_metrics, 'summary': summary, 'comparison_report': comparison_report}

def write_training_outputs(result, output_root):
    root = Path(output_root)
    root.mkdir(parents=True, exist_ok=True)
    for file_name in MANAGED_METADATA_FILES:
        path = root / file_name
        if path.exists():
            path.unlink()
    with (root / 'model.pkl').open('wb') as file_obj:
        pickle.dump(result['artifact_payload'], file_obj)
    (root / 'metrics.json').write_text(json.dumps(result['metrics'], indent=2), encoding='utf-8')
    (root / 'training_summary.json').write_text(json.dumps(result['summary'], indent=2), encoding='utf-8')
    (root / 'comparison_report.json').write_text(json.dumps(result['comparison_report'], indent=2), encoding='utf-8')
''')

train_code = textwrap.dedent('''
from __future__ import annotations
import argparse, json, os
from pathlib import Path
from ml.training.phase_c_train_multi import TrainingConfig, load_features_dataset, train_phase_c_models, write_training_outputs

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--features-path', default=os.environ.get('SM_CHANNEL_TRAINING', '/opt/ml/input/data/training'))
    parser.add_argument('--model-dir', default=os.environ.get('SM_MODEL_DIR', '/opt/ml/model'))
    parser.add_argument('--output-data-dir', default=os.environ.get('SM_OUTPUT_DATA_DIR', '/opt/ml/output/data'))
    parser.add_argument('--target-columns', default='')
    parser.add_argument('--validation-fraction', type=float, default=0.2)
    parser.add_argument('--min-validation-rows', type=int, default=14)
    parser.add_argument('--min-training-rows', type=int, default=30)
    parser.add_argument('--walk-forward-windows', type=int, default=3)
    return parser.parse_args()

def main():
    args = parse_args()
    config = TrainingConfig(validation_fraction=args.validation_fraction, min_validation_rows=args.min_validation_rows, min_training_rows=args.min_training_rows, walk_forward_windows=args.walk_forward_windows, target_columns=tuple(v.strip() for v in args.target_columns.split(',') if v.strip()))
    features = load_features_dataset(args.features_path)
    result = train_phase_c_models(features, config)
    model_dir = Path(args.model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    write_training_outputs(result, model_dir)
    output_dir = Path(args.output_data_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'training_report.json').write_text(json.dumps({'summary': result['summary'], 'metrics': result['metrics']}, indent=2), encoding='utf-8')

if __name__ == '__main__':
    main()
''')

evaluate_code = textwrap.dedent('''
from __future__ import annotations
import argparse, json, pickle
from pathlib import Path
from ml.training.phase_c_train_multi import TrainingConfig, build_multi_target_metric_summary, load_features_dataset, predict_multi_output, prepare_modeling_frame, resolve_target_columns, split_time_aware_dataset

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--features-path', required=True)
    parser.add_argument('--model-path', required=True)
    parser.add_argument('--output-path', required=True)
    parser.add_argument('--target-columns', default='')
    parser.add_argument('--validation-fraction', type=float, default=0.2)
    parser.add_argument('--min-validation-rows', type=int, default=14)
    parser.add_argument('--min-training-rows', type=int, default=30)
    return parser.parse_args()

def main():
    args = parse_args()
    config = TrainingConfig(validation_fraction=args.validation_fraction, min_validation_rows=args.min_validation_rows, min_training_rows=args.min_training_rows, target_columns=tuple(v.strip() for v in args.target_columns.split(',') if v.strip()))
    features = load_features_dataset(args.features_path)
    target_columns = resolve_target_columns(features, config)
    frame = prepare_modeling_frame(features, target_columns)
    train_frame, validation_frame = split_time_aware_dataset(frame, config)
    feature_columns = [c for c in frame.columns if c not in ('date', *target_columns)]
    with Path(args.model_path).open('rb') as file_obj:
        artifact = pickle.load(file_obj)
    predictions = predict_multi_output(artifact['model'], validation_frame, feature_columns, target_columns)
    metrics = build_multi_target_metric_summary(validation_frame[list(target_columns)], predictions, len(train_frame), len(validation_frame))
    output_path = Path(args.output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps({'evaluation_type': 'holdout', 'target_columns': list(target_columns), 'metrics': metrics}, indent=2), encoding='utf-8')

if __name__ == '__main__':
    main()
''')

inference_code = textwrap.dedent('''
from __future__ import annotations
import json, pickle
from pathlib import Path
import pandas as pd

def model_fn(model_dir):
    with Path(model_dir, 'model.pkl').open('rb') as file_obj:
        return pickle.load(file_obj)

def input_fn(request_body, content_type):
    if content_type != 'application/json':
        raise ValueError(f'Unsupported content type: {content_type}')
    payload = json.loads(request_body)
    if isinstance(payload, dict) and 'instances' in payload:
        return pd.DataFrame(payload['instances'])
    if isinstance(payload, list):
        return pd.DataFrame(payload)
    if isinstance(payload, dict):
        return pd.DataFrame([payload])
    raise ValueError('Unsupported JSON payload shape.')

def predict_fn(input_data, model_artifact):
    predictions = model_artifact['model'].predict(input_data[model_artifact['feature_columns']])
    return pd.DataFrame(predictions, columns=model_artifact['target_columns'])

def output_fn(prediction, accept):
    if accept != 'application/json':
        raise ValueError(f'Unsupported accept type: {accept}')
    return json.dumps({'predictions': prediction.to_dict(orient='records')})
''')


In [4]:
pipeline_definition_code = textwrap.dedent('''
from __future__ import annotations
import boto3
from sagemaker.inputs import TrainingInput
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.workflow.parameters import ParameterInteger, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.steps import ProcessingStep, TrainingStep

def build_pipeline(args):
    boto_session = boto3.session.Session(region_name=args.region)
    pipeline_session = PipelineSession(boto_session=boto_session, sagemaker_client=boto_session.client('sagemaker'), default_bucket=args.bucket)
    training_instance_type = ParameterString(name='TrainingInstanceType', default_value=args.default_instance_type)
    processing_instance_type = ParameterString(name='ProcessingInstanceType', default_value=args.default_instance_type)
    target_columns = ParameterString(name='TargetColumns', default_value='')
    train_input_s3_uri = ParameterString(name='FeaturesS3Uri', default_value=f's3://{args.bucket}/{args.features_prefix}')
    approval_status = ParameterString(name='ModelApprovalStatus', default_value='PendingManualApproval')
    validation_fraction = ParameterString(name='ValidationFraction', default_value='0.2')
    min_validation_rows = ParameterInteger(name='MinValidationRows', default_value=14)
    min_training_rows = ParameterInteger(name='MinTrainingRows', default_value=30)
    walk_forward_windows = ParameterInteger(name='WalkForwardWindows', default_value=3)
    estimator = SKLearn(entry_point='train.py', source_dir=args.code_bundle_s3_uri, role=args.role_arn, framework_version=args.framework_version, py_version=args.python_version, instance_type=training_instance_type, instance_count=1, sagemaker_session=pipeline_session, hyperparameters={'target-columns': target_columns, 'validation-fraction': validation_fraction, 'min-validation-rows': min_validation_rows, 'min-training-rows': min_training_rows, 'walk-forward-windows': walk_forward_windows}, output_path=f's3://{args.bucket}/sagemaker/training-output/')
    train_step = TrainingStep(name='TrainMultiOutputModel', estimator=estimator, inputs={'training': TrainingInput(s3_data=train_input_s3_uri, content_type='application/x-parquet')})
    evaluator = ScriptProcessor(image_uri=estimator.training_image_uri(), command=['python'], role=args.role_arn, instance_count=1, instance_type=processing_instance_type, sagemaker_session=pipeline_session)
    evaluation_report = PropertyFile(name='EvaluationReport', output_name='evaluation', path='evaluation.json')
    eval_step = ProcessingStep(name='EvaluateMultiOutputModel', processor=evaluator, code=args.evaluate_script_s3_uri, job_arguments=['--features-path', '/opt/ml/processing/input/features', '--model-path', '/opt/ml/processing/model/model.pkl', '--output-path', '/opt/ml/processing/evaluation/evaluation.json', '--target-columns', target_columns, '--validation-fraction', validation_fraction, '--min-validation-rows', str(min_validation_rows.default_value), '--min-training-rows', str(min_training_rows.default_value)], inputs=[ProcessingInput(source=train_input_s3_uri, destination='/opt/ml/processing/input/features'), ProcessingInput(source=train_step.properties.ModelArtifacts.S3ModelArtifacts, destination='/opt/ml/processing/model')], outputs=[ProcessingOutput(output_name='evaluation', source='/opt/ml/processing/evaluation')], property_files=[evaluation_report])
    model = SKLearnModel(model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts, role=args.role_arn, framework_version=args.framework_version, py_version=args.python_version, entry_point='inference.py', source_dir=args.code_bundle_s3_uri, sagemaker_session=pipeline_session)
    model_metrics = ModelMetrics(model_statistics=MetricsSource(s3_uri=eval_step.properties.ProcessingOutputConfig.Outputs['evaluation'].S3Output.S3Uri, content_type='application/json'))
    register_step = RegisterModel(name='RegisterMultiOutputModel', estimator=estimator, model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts, content_types=['application/json'], response_types=['application/json'], inference_instances=['ml.m5.large'], transform_instances=['ml.m5.large'], model_package_group_name=args.model_package_group_name, approval_status=approval_status, model_metrics=model_metrics)
    return Pipeline(name=args.pipeline_name, parameters=[training_instance_type, processing_instance_type, target_columns, train_input_s3_uri, approval_status, validation_fraction, min_validation_rows, min_training_rows, walk_forward_windows], steps=[train_step, eval_step, register_step], sagemaker_session=pipeline_session)
''')

In [5]:
paths = {
    'train.py': train_code,
    'evaluate.py': evaluate_code,
    'inference.py': inference_code,
    'pipeline_definition.py': pipeline_definition_code,
    'requirements.txt': 'numpy<2\npandas<2.2\nscikit-learn<1.4\nxgboost<2.1\npyarrow<15\n',
    'ml/training/phase_c_train_multi.py': phase_c_train_multi_code,
    'ml/__init__.py': '',
    'ml/training/__init__.py': '',
}
for relative_path, content in paths.items():
    path = WORKDIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
print('wrote files under', WORKDIR)

wrote files under /home/sagemaker-user/agri-price/agri_price_sagemaker_notebook_workdir


In [6]:
archive_path = WORKDIR / 'sagemaker_pipeline_source.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(WORKDIR / 'train.py', arcname='train.py')
    tar.add(WORKDIR / 'inference.py', arcname='inference.py')
    tar.add(WORKDIR / 'requirements.txt', arcname='requirements.txt')
    tar.add(WORKDIR / 'ml', arcname='ml')
s3.upload_file(str(archive_path), BUCKET, f'{CODE_PREFIX}sagemaker_pipeline_source.tar.gz')
s3.upload_file(str(WORKDIR / 'evaluate.py'), BUCKET, f'{CODE_PREFIX}evaluate.py')
print(CODE_BUNDLE_S3_URI)
print(EVALUATE_SCRIPT_S3_URI)

s3://agri-price-dev-raw/sagemaker/code/sagemaker_pipeline_source.tar.gz
s3://agri-price-dev-raw/sagemaker/code/evaluate.py


In [7]:
import sys
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
from pipeline_definition import build_pipeline

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [8]:
args = SimpleNamespace(region=region, role_arn=ROLE_ARN, bucket=BUCKET, features_prefix=FEATURES_PREFIX, pipeline_name=PIPELINE_NAME, model_package_group_name=MODEL_PACKAGE_GROUP_NAME, default_instance_type='ml.m5.xlarge', framework_version=FRAMEWORK_VERSION, python_version='py3', code_bundle_s3_uri=CODE_BUNDLE_S3_URI, evaluate_script_s3_uri=EVALUATE_SCRIPT_S3_URI)
pipeline = build_pipeline(args)
pipeline

In [9]:
try:
    sm.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME)
    print('model package group exists')
except ClientError as exc:
    if 'does not exist' in str(exc) or 'Could not find' in str(exc):
        sm.create_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME, ModelPackageGroupDescription='Multi-output agricultural price forecasting model')
        print('created model package group')
    else:
        raise

model package group exists


In [10]:
result = pipeline.upsert(role_arn=ROLE_ARN)
result

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:654654220088:pipeline/agri-price-train-evaluate-register',
 'ResponseMetadata': {'RequestId': '461a81da-bb61-4708-b26b-1656c3151803',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '461a81da-bb61-4708-b26b-1656c3151803',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '124',
   'date': 'Thu, 09 Apr 2026 16:49:45 GMT'},
  'RetryAttempts': 0}}

In [22]:
execution = pipeline.start(parameters={
    "TargetColumns": "target_next_day_price_coriander,target_next_day_price_kale,target_next_day_price_lime,target_next_day_price_orange,target_next_day_price_red_chili"
})
print(execution.arn)


arn:aws:sagemaker:us-east-1:654654220088:pipeline/agri-price-train-evaluate-register/execution/no986tt24319


In [26]:
executions = sm.list_pipeline_executions(PipelineName=PIPELINE_NAME)
executions

{'PipelineExecutionSummaries': [{'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:654654220088:pipeline/agri-price-train-evaluate-register/execution/no986tt24319',
   'StartTime': datetime.datetime(2026, 4, 9, 17, 13, 8, 387000, tzinfo=tzlocal()),
   'PipelineExecutionStatus': 'Failed',
   'PipelineExecutionDisplayName': 'execution-1775754788509',
   'PipelineExecutionFailureReason': 'Step failure: One or multiple steps failed.'},
  {'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:654654220088:pipeline/agri-price-train-evaluate-register/execution/2liak8tccs05',
   'StartTime': datetime.datetime(2026, 4, 9, 17, 4, 14, 718000, tzinfo=tzlocal()),
   'PipelineExecutionStatus': 'Failed',
   'PipelineExecutionDisplayName': 'execution-1775754254802',
   'PipelineExecutionFailureReason': 'Step failure: One or multiple steps failed.'},
  {'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:654654220088:pipeline/agri-price-train-evaluate-register/execution/ktkx9i6e3y16',
   'StartTime'

In [27]:
steps = sm.list_pipeline_execution_steps(
    PipelineExecutionArn='arn:aws:sagemaker:us-east-1:654654220088:pipeline/agri-price-train-evaluate-register/execution/no986tt24319'
)
steps['PipelineExecutionSteps']


[{'StepName': 'EvaluateMultiOutputModel',
  'StartTime': datetime.datetime(2026, 4, 9, 17, 16, 18, 122000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 4, 9, 17, 18, 51, 121000, tzinfo=tzlocal()),
  'StepStatus': 'Failed',
  'FailureReason': 'ClientError: AlgorithmError: , exit code: 1',
  'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:654654220088:processing-job/pipelines-no986tt24319-EvaluateMultiOutputM-dszJajHzJQ'}},
  'AttemptCount': 1},
 {'StepName': 'TrainMultiOutputModel',
  'StartTime': datetime.datetime(2026, 4, 9, 17, 13, 9, 133000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 4, 9, 17, 16, 17, 383000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'TrainingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:654654220088:training-job/pipelines-no986tt24319-TrainMultiOutputMode-LeTwG59l0d'}},
  'AttemptCount': 1}]

In [15]:
latest_execution_arn = executions['PipelineExecutionSummaries'][0]['PipelineExecutionArn']
steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=latest_execution_arn)
steps

{'PipelineExecutionSteps': [{'StepName': 'TrainMultiOutputModel',
   'StartTime': datetime.datetime(2026, 4, 9, 14, 45, 53, 164000, tzinfo=tzlocal()),
   'EndTime': datetime.datetime(2026, 4, 9, 14, 48, 11, 647000, tzinfo=tzlocal()),
   'StepStatus': 'Failed',
   'FailureReason': 'ClientError: AlgorithmError: framework error: \nTraceback (most recent call last):\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_trainer.py", line 84, in train\n    entrypoint()\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 39, in main\n    train(environment.Environment())\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 31, in train\n    entry_point.run(uri=training_environment.module_dir,\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/entry_point.py", line 108, in run\n    return runner.get(runner_type, user_entry_point, args, env_vars, extra_opts).run(\n  File "/min